In [ ]:
import pandas as pd
import re


# Creación de la base de datos

In [ ]:
comprobantes = pd.read_excel('Comprobantes detallados 2025 SIIGO.xlsx', header=7)
comprobantes.head(10)

In [ ]:
# Extraer código de comprobante y propagarlo
comprobantes['Comprobante'] = comprobantes['Secuencia'].where(
    comprobantes['Secuencia'].astype(str).str.startswith('Comprobante:')
).ffill()
comprobantes['Comprobante'] = comprobantes['Comprobante'].str.replace('Comprobante: ', '', regex=False)

# Eliminar filas encabezado de comprobante
comprobantes = comprobantes[
    ~comprobantes['Secuencia'].astype(str).str.startswith('Comprobante:')
].reset_index(drop=True)

# Convertir columnas numéricas (Int64 soporta NaN)
comprobantes['Código contable'] = comprobantes['Código contable'].astype('Int64')
comprobantes['Identificación']  = comprobantes['Identificación'].astype('Int64')
comprobantes['Sucursal']        = comprobantes['Sucursal'].astype('Int64')

# Movimiento: crédito cuando débito=0, -débito cuando crédito=0
comprobantes['Movimiento'] = comprobantes.apply(
    lambda r: r['Crédito'] if r['Débito'] == 0 else -r['Débito'], axis=1
)

cols = ['Comprobante'] + [c for c in comprobantes.columns if c != 'Comprobante']
comprobantes = comprobantes[cols]
comprobantes.head(20)

In [ ]:
productos = pd.read_excel('Listado de productos _ Servicios .xlsx', header=6)

# Filtrar filas válidas
productos = productos[productos['Tipo'].isin(['Producto', 'Servicio'])].copy().reset_index(drop=True)

# Renombrar para consistencia
productos = productos.rename(columns={'Código': 'Código producto', 'Nombre': 'Nombre producto'})

print(f'Productos en catálogo: {len(productos)}')
productos.head(10)

In [ ]:
def norm_nombre_det(s):
    """Normaliza nombre extraído del campo Detalle del comprobante."""
    if pd.isna(s): return s
    s = str(s).strip()
    s = re.sub(r'\s+', ' ', s)                    # normalizar espacios múltiples
    s = re.sub(r'\s+Bod:.*$', '', s)               # quitar info de bodega
    s = re.sub(r'\s+M\.?\s*GEN(?:E(?:RICA)?)?$', '', s, flags=re.IGNORECASE)
    s = re.sub(r'\s+M\.?\s*(CENTELSA|CONDUMEX|SYLVANIA|SULVANIA|MAXWELL|KLIC|MERCURY|LEGRAND|TUBOSA)$',
               '', s, flags=re.IGNORECASE)
    parts = s.split()
    if len(parts) > 1 and re.match(r'^[A-Z]{3,}$', parts[-1]):
        s = ' '.join(parts[:-1])
    return s.rstrip('. ')

def norm_nombre_cat(s):
    """Normaliza nombre del catálogo de productos."""
    if pd.isna(s): return s
    s = str(s).strip()
    s = re.sub(r'\s+M\.?\s*GEN(?:E(?:RICA)?)?$', '', s, flags=re.IGNORECASE)
    s = re.sub(r'\s+M\.?\s+[A-Z]+$', '', s, flags=re.IGNORECASE)
    s = re.sub(r'\s+(MAXWELL|CENTELSA|CONDUMEX|SYLVANIA|SULVANIA|TUBOSA|KLIC|MERCURY|LEGRAND|PHILLIPS)$',
               '', s, flags=re.IGNORECASE)
    return s.rstrip('. ')

productos['nombre_norm'] = productos['Nombre producto'].apply(norm_nombre_cat)

# Lookups
_prod_dedup  = productos.drop_duplicates('Código producto')
lookup_code  = _prod_dedup.set_index('Código producto')['Categoría']
lookup_exact = productos.drop_duplicates('Nombre producto').set_index('Nombre producto')['Código producto']
lookup_norm  = productos.drop_duplicates('nombre_norm').set_index('nombre_norm')['Código producto']

tiene_prod     = comprobantes['Detalle'].str.contains(r'Prod(?:ucto)?:', na=False)
total_con_prod = tiene_prod.sum()

# Extraer código y nombre desde Detalle
comprobantes['_codigo_det'] = comprobantes['Detalle'].str.extract(r'Prod:\s*(\S+)')
comprobantes['_nombre_det'] = comprobantes['Detalle'].str.extract(r'Producto:\s*(.+?)\s*Cant:', flags=re.IGNORECASE)
comprobantes['Código producto'] = pd.NA
comprobantes['Categoría']       = pd.NA

# --- Paso 1: Código directo (Prod: CODIGO) ---
mask1 = comprobantes['_codigo_det'].notna() & tiene_prod
comprobantes.loc[mask1, 'Código producto'] = comprobantes.loc[mask1, '_codigo_det']
comprobantes.loc[mask1, 'Categoría'] = comprobantes.loc[mask1, '_codigo_det'].map(lookup_code)

# --- Paso 2: Nombre exacto (Producto: NOMBRE) ---
sin2 = comprobantes['Código producto'].isna() & tiene_prod & comprobantes['_nombre_det'].notna()
m2   = comprobantes.loc[sin2, '_nombre_det'].map(lookup_exact)
idx2 = m2.index[m2.notna()]
comprobantes.loc[idx2, 'Código producto'] = m2[m2.notna()].values
comprobantes.loc[idx2, 'Categoría'] = comprobantes.loc[idx2, 'Código producto'].map(lookup_code)

# --- Paso 3: Nombre normalizado ---
sin3     = comprobantes['Código producto'].isna() & tiene_prod & comprobantes['_nombre_det'].notna()
det_norm = comprobantes.loc[sin3, '_nombre_det'].apply(norm_nombre_det)
m3       = det_norm.map(lookup_norm)
idx3     = m3.index[m3.notna()]
comprobantes.loc[idx3, 'Código producto'] = m3[m3.notna()].values
comprobantes.loc[idx3, 'Categoría'] = comprobantes.loc[idx3, 'Código producto'].map(lookup_code)

# Reporte
sin_final     = comprobantes['Código producto'].isna() & tiene_prod
total_matched = total_con_prod - sin_final.sum()
print(f'Paso 1 - Código directo:  {mask1.sum():5d} ({mask1.sum()/total_con_prod*100:.1f}%)')
print(f'Paso 2 - Nombre exacto:   {m2.notna().sum():5d} ({m2.notna().sum()/total_con_prod*100:.1f}%)')
print(f'Paso 3 - Nombre norm:     {m3.notna().sum():5d} ({m3.notna().sum()/total_con_prod*100:.1f}%)')
print(f'TOTAL matched:            {total_matched:5d} / {total_con_prod} ({total_matched/total_con_prod*100:.1f}%)')
print(f'DESCONOCIDO:              {sin_final.sum():5d} ({sin_final.sum()/total_con_prod*100:.1f}%)')

if sin_final.any():
    restantes = sorted(comprobantes.loc[sin_final, '_nombre_det'].dropna().unique())
    print(f'\nDescripciones sin match ({len(restantes)}):')
    for d in restantes: print(f'  - {d}')

# Asignar DESCONOCIDO y limpiar columnas auxiliares
comprobantes.loc[comprobantes['Código producto'].isna() & tiene_prod, 'Código producto'] = 'DESCONOCIDO'
comprobantes = comprobantes.drop(columns=['_codigo_det', '_nombre_det'])
comprobantes.head(10)

# Procesamiento de datos

In [ ]:
# Facturas de venta (FV) con cuenta de ingresos (41350101)
ventas = comprobantes[
    (comprobantes['Comprobante'].str.startswith('FV')) &
    (comprobantes['Código contable'] == 41350101)
].copy()

# Enriquecer con nombre del producto desde catálogo
ventas = ventas.merge(
    productos[['Código producto', 'Nombre producto']].drop_duplicates('Código producto'),
    on='Código producto', how='left'
)

# Columnas de fecha y tiempo
ventas['Fecha']      = pd.to_datetime(ventas['Fecha elaboración'], dayfirst=True)
ventas['Año']        = ventas['Fecha'].dt.year
ventas['Trimestre']  = ventas['Fecha'].dt.quarter
ventas['Mes']        = ventas['Fecha'].dt.month
ventas['Mes_nombre'] = ventas['Fecha'].dt.strftime('%b')
ventas['Semana']     = ventas['Fecha'].dt.isocalendar().week.astype(int)

# Cantidad vendida
ventas['Cantidad'] = ventas['Detalle'].str.extract(r'Cant:\s*([\d.]+)').astype(float)

print(f'Filas de ventas: {len(ventas)}')
print(f'Facturas únicas: {ventas["Comprobante"].nunique()}')
print(f'Clientes únicos: {ventas["Nombre tercero"].nunique()}')
print(f'Productos únicos: {ventas["Código producto"].nunique()}')
print(f'Venta total: ${ventas["Crédito"].sum():,.0f}')
ventas.head()

In [ ]:
# --- comprobantes_enriquecidos.csv ---
# Todos los movimientos contables con producto y categoría
comprobantes_out = comprobantes[[
    'Comprobante', 'Fecha elaboración', 'Código contable', 'Cuenta contable',
    'Identificación', 'Nombre tercero', 'Descripción', 'Detalle',
    'Débito', 'Crédito', 'Movimiento',
    'Código producto', 'Categoría'
]]
comprobantes_out.to_csv('comprobantes_enriquecidos.csv', index=False)

# --- ventas.csv ---
# Solo facturas de venta, con columnas de fecha y producto para dashboards
ventas_out = ventas[[
    'Comprobante', 'Fecha', 'Año', 'Trimestre', 'Mes', 'Mes_nombre', 'Semana',
    'Identificación', 'Nombre tercero',
    'Código producto', 'Nombre producto', 'Categoría',
    'Descripción', 'Cantidad', 'Crédito'
]].rename(columns={'Crédito': 'Venta'})
ventas_out.to_csv('ventas.csv', index=False)

# --- productos.csv ---
productos.drop(columns=['nombre_norm'], errors='ignore').to_csv('productos.csv', index=False)

print(f'comprobantes_enriquecidos.csv  → {len(comprobantes_out):,} filas')
print(f'ventas.csv                     → {len(ventas_out):,} filas')
print(f'productos.csv                  → {len(productos):,} filas')
print()
print('Columnas ventas.csv:')
print(ventas_out.dtypes.to_string())